# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available Record Sets and their @id
print("Available Record Sets:")
for rec in metadata.record_sets:
    print(f"- Name: {rec.name}, @id: {rec.id}")

# For each record set, list available fields and their @id
for rec in metadata.record_sets:
    print(f"\nRecord Set: {rec.name} (@id: {rec.id})")
    print("Fields:")
    for field in rec.fields:
        print(f"  - {field.name} (@id: {field.id}, data type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into a DataFrame
record_sets = [rec.id for rec in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# For demonstration, select the first record set for deeper analysis
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Selected RecordSet for analysis: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    print("\nColumns in main record set:")
    print(df.columns.tolist())

    # Try to find a numeric field for demonstration
    # We'll prefer columns named with typical quantitative names
    preferred_numeric_fields = ['MW', 'molecular_weight', 'Abundance', 'abundance', 'Number_Peptides', 'num_peptides', 'Coverage', 'coverage', 'NormAbundance']
    numeric_field = None
    for col in df.columns:
        if col in preferred_numeric_fields and np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    # If none found, pick the first numeric column
    if numeric_field is None:
        for col in df.columns:
            try:
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        # Impute missing values for demonstration
        df = df.copy()
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if df[numeric_field].dtype in [float, int, np.float64, np.int64] else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        if filtered_df[numeric_field].std() != 0:
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical/group field
        group_field = None
        preferred_group_fields = ['description', 'Accession', 'ProteinID', 'Sample', 'sample', 'Category']
        for col in df.columns:
            if col in preferred_group_fields and col != numeric_field:
                group_field = col
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found in the main record set.")
else:
    print("No available record sets for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If normalized column exists (from previous step)
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.xlabel(f"{numeric_field} (normalized)")
        plt.title(f'Normalized Distribution for Filtered {numeric_field}')
        plt.show()

    # Boxplot by group field if exists
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the FAIR^2 dataset, which comprises protein abundance and modification records from mass spectrometry analysis of extracellular vesicles derived from stimulated human mast cells. 

- We identified all available record sets and fields, referencing them by their `@id` as per Croissant standards.
- Data from a selected record set was loaded into a pandas DataFrame, filtered by a numeric field, normalized, and grouped by a categorical attribute where possible.
- We visualized the distribution of a key quantitative parameter to reveal data characteristics and potential patterns.

**You can now build on this notebook for domain-specific analysis or advanced machine learning tasks using the rich, semantically described Croissant dataset.**